# 🛡️ AI-Based Intrusion Detection & Automated Response System (IDRS)
## For Educational Web Platforms — Tunisia Context
---
## PART 2: Preprocessing Pipeline · Classical ML Baselines · Optuna Tuning · SHAP Explainability

**Depends on:** `IDRS_Part1_Setup_EDA.ipynb` → `outputs/idrs_processed_part1.parquet`

**Scope of Part 2:**
- ✅ Full reproducible preprocessing pipeline (scaling, encoding, SMOTE)
- ✅ Stratified time-aware train / val / test split
- ✅ Three classical ML baselines: Random Forest · XGBoost · LightGBM
- ✅ Optuna Bayesian hyperparameter optimisation (per model)
- ✅ Threshold tuning for maximum recall on critical attack classes
- ✅ Full evaluation suite: accuracy · F1 · ROC-AUC · PR-AUC · confusion matrix
- ✅ SHAP global + local explainability
- ✅ Model persistence (joblib) → ready for Part 3 ensemble

---
### ⚙️ Quick-start
```
outputs/
  idrs_processed_part1.parquet   ← input from Part 1
  label_classes.json             ← input from Part 1
models/
  rf_best.joblib                 ← output
  xgb_best.joblib                ← output
  lgbm_best.joblib               ← output
  preprocessor.joblib            ← output (scaler + column order)
  thresholds.json                ← output (per-class optimal thresholds)
```

---
## ⚙️ CELL 1 — Imports & Config

In [ ]:
# ============================================================
# IDRS Part 2 — Imports & Configuration
# ============================================================
import os, sys, json, warnings, logging, time
from pathlib import Path
from datetime import datetime
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────
from sklearn.pipeline          import Pipeline
from sklearn.preprocessing     import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection   import (StratifiedKFold, train_test_split,
                                        cross_val_score, StratifiedShuffleSplit)
from sklearn.ensemble          import RandomForestClassifier
from sklearn.metrics           import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, roc_curve, precision_recall_curve,
    balanced_accuracy_score, matthews_corrcoef
)
from sklearn.utils             import class_weight as sk_class_weight
from sklearn.inspection        import permutation_importance

# ── Imbalanced-learn ─────────────────────────────────────
from imblearn.over_sampling    import SMOTE, BorderlineSMOTE, SVMSMOTE
from imblearn.combine          import SMOTETomek
from imblearn.pipeline         import Pipeline as ImbPipeline

# ── Tree-based models ─────────────────────────────────────
import xgboost  as xgb
import lightgbm as lgb

# ── Hyperparameter optimisation ───────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Explainability ────────────────────────────────────────
import shap
shap.initjs()

# ── Utilities ─────────────────────────────────────────────
import joblib
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Directory paths ───────────────────────────────────────
BASE_DIR   = Path(".")
OUTPUT_DIR = BASE_DIR / "outputs"
MODEL_DIR  = BASE_DIR / "models"
LOG_DIR    = BASE_DIR / "logs"
for d in [OUTPUT_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Logging ──────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = '%(asctime)s | %(levelname)s | %(message)s',
    handlers= [
        logging.FileHandler(LOG_DIR / "idrs_part2.log"),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger("IDRS-P2")

# ── Plot style ────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'font.size': 11, 'axes.titlesize': 13, 'figure.dpi': 120,
})

PALETTE = {
    'BENIGN':'#2ecc71','DoS':'#e74c3c','DDoS':'#c0392b',
    'Probe':'#f39c12','WebAttack':'#9b59b6','BruteForce':'#3498db',
    'Botnet':'#1abc9c','Exploit':'#e67e22','UNKNOWN':'#95a5a6'
}

logger.info("Part 2 environment ready.")
print("✅ Part 2 environment ready.")

---
## 📥 CELL 2 — Load Part 1 Outputs

In [ ]:
# ============================================================
# IDRS Part 2 — Load processed dataset from Part 1
# ============================================================

PARQUET_PATH = OUTPUT_DIR / 'idrs_processed_part1.parquet'
LABELS_PATH  = OUTPUT_DIR / 'label_classes.json'

# ── Load dataset ──────────────────────────────────────────
if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    logger.info("Loaded Part 1 dataset: %d rows × %d cols", *df.shape)
else:
    # Regenerate synthetic data if Part 1 not yet run
    logger.warning("Part 1 parquet not found. Re-generating synthetic data.")
    print("⚠️  Part 1 output not found. Running synthetic fallback...")
    # (Paste generate_synthetic_idrs_data from Part 1 here if needed)
    raise FileNotFoundError(
        f"Run Part 1 first to generate: {PARQUET_PATH}\n"
        "Or re-paste the synthetic generator from Part 1 Cell 5."
    )

# ── Load label classes ────────────────────────────────────
if LABELS_PATH.exists():
    with open(LABELS_PATH) as f:
        label_classes = json.load(f)
    INT_TO_CLASS = {int(k): v for k, v in label_classes.items()}
    CLASS_TO_INT = {v: k for k, v in INT_TO_CLASS.items()}
    CLASS_NAMES  = [INT_TO_CLASS[i] for i in sorted(INT_TO_CLASS.keys())]
else:
    le = LabelEncoder()
    le.fit(df['threat_class'])
    CLASS_NAMES  = list(le.classes_)
    CLASS_TO_INT = {c: i for i, c in enumerate(CLASS_NAMES)}
    INT_TO_CLASS = {i: c for c, i in CLASS_TO_INT.items()}

N_CLASSES = len(CLASS_NAMES)

# ── Verify label_encoded exists ───────────────────────────
if 'label_encoded' not in df.columns:
    df['label_encoded'] = df['threat_class'].map(CLASS_TO_INT)

print(f"✅ Dataset loaded: {len(df):,} rows × {df.shape[1]} columns")
print(f"   Threat classes ({N_CLASSES}): {CLASS_NAMES}")
print(f"\n   Class distribution:")
for c, n in df['threat_class'].value_counts().items():
    pct = n / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"   {c:<14} {n:>8,}  {pct:5.1f}%  {bar}")

---
## 🔧 CELL 3 — Feature Matrix Construction & Column Audit

In [ ]:
# ============================================================
# IDRS Part 2 — Build Feature Matrix X and Target Vector y
# ============================================================

# ── Define non-feature columns ────────────────────────────
NON_FEATURE_COLS = [
    'label', 'threat_class', 'dataset', 'source_file',
    'label_encoded', 'attack_cat',
    # UNSW-NB15 IP/port columns (identifiers, not features)
    'srcip', 'dstip', 'sport', 'dsport', 'Stime', 'Ltime',
]

# ── Select numeric feature columns ────────────────────────
FEATURE_COLS = [
    c for c in df.columns
    if c not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(df[c])
    and df[c].nunique() > 1          # remove zero-variance
]

print(f"📐 Feature matrix: {len(FEATURE_COLS)} features")

X = df[FEATURE_COLS].copy()
y = df['label_encoded'].values

# ── Final NaN/Inf guard ───────────────────────────────────
X = X.replace([np.inf, -np.inf], np.nan)
nan_medians = X.median()
X = X.fillna(nan_medians)
X = X.clip(lower=-1e12, upper=1e12)  # clip extreme outliers

assert not X.isna().any().any(), "NaNs remain after imputation!"
assert not np.isinf(X.values).any(), "Infs remain after clipping!"

# ── Feature group summary ─────────────────────────────────
groups = {
    'Flow statistics'   : [c for c in FEATURE_COLS if 'Flow' in c],
    'Packet lengths'    : [c for c in FEATURE_COLS if 'Packet Length' in c or 'Pkt' in c],
    'Flag counts'       : [c for c in FEATURE_COLS if 'Flag' in c],
    'IAT features'      : [c for c in FEATURE_COLS if 'IAT' in c],
    'Engineered (feat_)': [c for c in FEATURE_COLS if c.startswith('feat_')],
    'Other'             : [c for c in FEATURE_COLS
                           if not any(k in c for k in
                                      ['Flow','Packet Length','Flag','IAT','feat_'])],
}

print("\n   Feature groups:")
for grp, cols in groups.items():
    if cols:
        print(f"   {grp:<25} {len(cols):>3} features")

print(f"\n   Target: y shape={y.shape}, classes={np.unique(y)}")
print(f"   X dtype summary: {X.dtypes.value_counts().to_dict()}")

---
## ✂️ CELL 4 — Stratified Train / Validation / Test Split

In [ ]:
# ============================================================
# IDRS Part 2 — Stratified Train / Val / Test Split
#
# Strategy:
#   70% train  (SMOTE applied here only)
#   15% val    (clean, for threshold tuning)
#   15% test   (clean, held-out final evaluation)
#
# Stratified to preserve class ratios in all splits.
# Time-aware: if source_file present, test = latest day's data.
# ============================================================

USE_TEMPORAL_SPLIT = (
    'source_file' in df.columns and
    df['source_file'].nunique() > 3 and
    df['source_file'].str.lower().str.contains('friday|thursday|wednesday').any()
)

if USE_TEMPORAL_SPLIT:
    # ── Temporal split: test = latest day ─────────────────
    days_ordered = sorted(df['source_file'].unique())
    test_day     = days_ordered[-1]
    test_mask    = df['source_file'] == test_day

    X_trainval = X[~test_mask].values
    y_trainval = y[~test_mask]
    X_test     = X[test_mask].values
    y_test     = y[test_mask]

    # Split trainval → train + val
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size    = 0.15 / 0.85,
        stratify     = y_trainval,
        random_state = SEED
    )
    split_type = f"Temporal (test='{test_day}')"

else:
    # ── Stratified random split ────────────────────────────
    X_arr = X.values

    X_temp, X_test, y_temp, y_test = train_test_split(
        X_arr, y, test_size=0.15, stratify=y, random_state=SEED
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size    = 0.15 / 0.85,
        stratify     = y_temp,
        random_state = SEED
    )
    split_type = "Stratified random"

print(f"✅ Split strategy : {split_type}")
print(f"   Train   : {len(X_train):>8,}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"   Val     : {len(X_val):>8,}  ({len(X_val)/len(X)*100:.1f}%)")
print(f"   Test    : {len(X_test):>8,}  ({len(X_test)/len(X)*100:.1f}%)")

# ── Verify class presence in each split ──────────────────
print("\n   Class coverage per split:")
for split_name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    missing = set(range(N_CLASSES)) - set(np.unique(y_split))
    present = len(np.unique(y_split))
    status  = "✅" if not missing else f"⚠️  missing: {[INT_TO_CLASS[m] for m in missing]}"
    print(f"   {split_name:<8}: {present}/{N_CLASSES} classes  {status}")

---
## ⚖️ CELL 5 — Scaling & SMOTE Oversampling Pipeline

In [ ]:
# ============================================================
# IDRS Part 2 — Preprocessing: Scaling + SMOTE
#
# Pipeline:
#   1. StandardScaler (fit on train, transform all splits)
#   2. SMOTE strategy selection based on imbalance ratio
#      - Ratio < 5x   → no resampling needed
#      - 5x–20x       → BorderlineSMOTE (focus boundary cases)
#      - >20x          → SMOTETomek (oversample + clean noise)
#
# CRITICAL: SMOTE applied ONLY to training set.
# Val and Test remain pristine real-world distributions.
# ============================================================

# ── 1. Standard Scaling ───────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

logger.info("Scaler fitted. Mean range: [%.4f, %.4f]",
            scaler.mean_.min(), scaler.mean_.max())

# ── 2. Compute imbalance ratio on training set ────────────
train_counts  = pd.Series(y_train).value_counts()
imbalance_ratio = train_counts.max() / train_counts.min()

print(f"📊 Training set class distribution (before SMOTE):")
for cls_id, cnt in train_counts.sort_index().items():
    cls_name = INT_TO_CLASS.get(cls_id, str(cls_id))
    bar = '█' * int(cnt / train_counts.max() * 30)
    print(f"   {cls_name:<14} {cnt:>7,}  {bar}")
print(f"\n   Imbalance ratio: {imbalance_ratio:.1f}x")

# ── 3. SMOTE Strategy Selection ──────────────────────────
print("\n⚖️  Selecting SMOTE strategy...")

# Minimum samples needed to apply SMOTE (k_neighbors + 1)
K_NEIGHBORS = 5
min_class_n = train_counts.min()

# Build sampling_strategy: bring minority classes up to
# at most 20% of the majority class count
majority_n  = train_counts.max()
target_n    = max(int(majority_n * 0.20), min_class_n)
sampling_strategy = {
    cls_id: max(target_n, cnt)
    for cls_id, cnt in train_counts.items()
    if cnt < target_n and cnt > K_NEIGHBORS
}

if not sampling_strategy or imbalance_ratio < 3:
    print("   → Imbalance ratio < 3. No resampling needed.")
    X_train_res, y_train_res = X_train_scaled, y_train
    SMOTE_APPLIED = False

elif imbalance_ratio <= 20:
    print(f"   → Moderate imbalance ({imbalance_ratio:.1f}x). Applying BorderlineSMOTE...")
    print(f"   → Target per minority class: {target_n:,}")
    smote = BorderlineSMOTE(
        sampling_strategy = sampling_strategy,
        k_neighbors       = K_NEIGHBORS,
        random_state      = SEED,
        kind              = 'borderline-1',
    )
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
    SMOTE_APPLIED = True
    SMOTE_TYPE    = "BorderlineSMOTE"

else:
    print(f"   → Severe imbalance ({imbalance_ratio:.1f}x). Applying SMOTETomek...")
    print(f"   → Target per minority class: {target_n:,}")
    smote = SMOTETomek(
        smote             = SMOTE(
            sampling_strategy = sampling_strategy,
            k_neighbors       = K_NEIGHBORS,
            random_state      = SEED
        ),
        random_state      = SEED,
    )
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
    SMOTE_APPLIED = True
    SMOTE_TYPE    = "SMOTETomek"

# ── Post-SMOTE report ─────────────────────────────────────
if SMOTE_APPLIED:
    after_counts = pd.Series(y_train_res).value_counts().sort_index()
    new_ratio    = after_counts.max() / after_counts.min()
    print(f"\n   ✅ {SMOTE_TYPE} applied.")
    print(f"   Training set: {len(X_train):,} → {len(X_train_res):,} samples")
    print(f"   New imbalance ratio: {new_ratio:.1f}x  (was {imbalance_ratio:.1f}x)")
    print(f"\n   After-SMOTE distribution:")
    for cls_id, cnt in after_counts.items():
        cls_name = INT_TO_CLASS.get(cls_id, str(cls_id))
        bar = '█' * int(cnt / after_counts.max() * 30)
        print(f"   {cls_name:<14} {cnt:>7,}  {bar}")

# ── Save preprocessor artifacts ──────────────────────────
preprocessor_meta = {
    'feature_cols'    : FEATURE_COLS,
    'n_features'      : len(FEATURE_COLS),
    'n_classes'       : N_CLASSES,
    'class_names'     : CLASS_NAMES,
    'smote_applied'   : SMOTE_APPLIED,
    'smote_type'      : SMOTE_TYPE if SMOTE_APPLIED else 'none',
    'split_type'      : split_type,
    'train_n'         : len(X_train_res),
    'val_n'           : len(X_val_scaled),
    'test_n'          : len(X_test_scaled),
    'scaler_mean_clip': [float(scaler.mean_.min()), float(scaler.mean_.max())],
}
joblib.dump(scaler,               MODEL_DIR / 'scaler.joblib')
with open(MODEL_DIR / 'preprocessor_meta.json', 'w') as f:
    json.dump(preprocessor_meta, f, indent=2)

# Compute class weights for loss-weighted models
CW = sk_class_weight.compute_class_weight(
    class_weight = 'balanced',
    classes      = np.unique(y_train_res),
    y            = y_train_res
)
CLASS_WEIGHT_DICT = dict(enumerate(CW))

print(f"\n💾 Saved: models/scaler.joblib + models/preprocessor_meta.json")
print(f"   Class weight dict: {CLASS_WEIGHT_DICT}")

---
## 📈 CELL 6 — Evaluation Toolkit

In [ ]:
# ============================================================
# IDRS Part 2 — Reusable Evaluation Toolkit
# ============================================================

def evaluate_model(
    model,
    X_eval     : np.ndarray,
    y_eval     : np.ndarray,
    model_name : str = "Model",
    split_name : str = "Test",
    class_names: List[str] = None,
    plot       : bool = True,
    save_prefix: str = "",
) -> Dict[str, Any]:
    """
    Compute and optionally plot a full evaluation suite:
      - Accuracy, balanced accuracy, MCC
      - Per-class precision, recall, F1
      - Macro / weighted averages
      - ROC-AUC (OvR) and PR-AUC
      - Confusion matrix (normalised)
      - Per-class ROC and PR curves
    """
    class_names = class_names or CLASS_NAMES
    n_cls       = len(class_names)

    y_pred      = model.predict(X_eval)
    y_prob      = model.predict_proba(X_eval)   # shape: (n, n_classes)

    acc   = accuracy_score(y_eval, y_pred)
    bacc  = balanced_accuracy_score(y_eval, y_pred)
    mcc   = matthews_corrcoef(y_eval, y_pred)
    f1_mac= f1_score(y_eval, y_pred, average='macro', zero_division=0)
    f1_wgt= f1_score(y_eval, y_pred, average='weighted', zero_division=0)
    prec_mac = precision_score(y_eval, y_pred, average='macro', zero_division=0)
    rec_mac  = recall_score(y_eval, y_pred, average='macro', zero_division=0)

    # ROC-AUC (one-vs-rest, macro)
    y_bin = label_binarize(y_eval, classes=list(range(n_cls)))
    try:
        roc_auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
    except Exception:
        roc_auc = float('nan')

    # PR-AUC (macro average)
    pr_aucs = []
    for c in range(n_cls):
        if y_bin[:, c].sum() > 0:
            pr_aucs.append(average_precision_score(y_bin[:, c], y_prob[:, c]))
    pr_auc = float(np.mean(pr_aucs)) if pr_aucs else float('nan')

    results = {
        'model'      : model_name,
        'split'      : split_name,
        'accuracy'   : round(acc,    4),
        'balanced_acc': round(bacc,  4),
        'mcc'        : round(mcc,    4),
        'f1_macro'   : round(f1_mac, 4),
        'f1_weighted': round(f1_wgt, 4),
        'precision_macro': round(prec_mac, 4),
        'recall_macro'   : round(rec_mac,  4),
        'roc_auc'    : round(roc_auc, 4),
        'pr_auc'     : round(pr_auc,  4),
    }

    print(f"\n{'─'*58}")
    print(f"  {model_name} — {split_name} Set Evaluation")
    print(f"{'─'*58}")
    print(f"  Accuracy          : {acc:.4f}")
    print(f"  Balanced Accuracy : {bacc:.4f}")
    print(f"  MCC               : {mcc:.4f}")
    print(f"  F1 Macro          : {f1_mac:.4f}")
    print(f"  F1 Weighted       : {f1_wgt:.4f}")
    print(f"  ROC-AUC (macro)   : {roc_auc:.4f}")
    print(f"  PR-AUC  (macro)   : {pr_auc:.4f}")
    print(f"\n{classification_report(y_eval, y_pred, target_names=class_names, zero_division=0)}")

    if not plot:
        return results

    fig = plt.figure(figsize=(22, 9))
    gs  = GridSpec(1, 3, figure=fig, wspace=0.35)

    # ── 1. Confusion matrix ──────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    cm  = confusion_matrix(y_eval, y_pred, normalize='true')
    sns.heatmap(
        cm, ax=ax1, annot=True, fmt='.2f',
        xticklabels=class_names, yticklabels=class_names,
        cmap='Blues', vmin=0, vmax=1,
        linewidths=0.5, cbar_kws={'label': 'Recall'},
        annot_kws={'size': 7}
    )
    ax1.set_xlabel('Predicted', fontsize=9)
    ax1.set_ylabel('True', fontsize=9)
    ax1.set_title(f'Confusion Matrix\n{model_name} · {split_name}', fontweight='bold')
    ax1.tick_params(axis='x', rotation=35, labelsize=7)
    ax1.tick_params(axis='y', rotation=0, labelsize=7)

    # ── 2. Per-class ROC curves ──────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    for c in range(n_cls):
        if y_bin[:, c].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, c], y_prob[:, c])
        auc_c = roc_auc_score(y_bin[:, c], y_prob[:, c])
        color = PALETTE.get(class_names[c], '#95a5a6')
        ax2.plot(fpr, tpr, lw=1.8, color=color,
                 label=f"{class_names[c]} (AUC={auc_c:.3f})")
    ax2.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random')
    ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
    ax2.set_title(f'ROC Curves (per class)\n{model_name}', fontweight='bold')
    ax2.legend(fontsize=7, loc='lower right')
    ax2.set_xlim([-0.01, 1.01]); ax2.set_ylim([-0.01, 1.01])

    # ── 3. Per-class PR curves ───────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    for c in range(n_cls):
        if y_bin[:, c].sum() == 0:
            continue
        prec, rec, _ = precision_recall_curve(y_bin[:, c], y_prob[:, c])
        ap = average_precision_score(y_bin[:, c], y_prob[:, c])
        color = PALETTE.get(class_names[c], '#95a5a6')
        ax3.plot(rec, prec, lw=1.8, color=color,
                 label=f"{class_names[c]} (AP={ap:.3f})")
    ax3.set_xlabel('Recall'); ax3.set_ylabel('Precision')
    ax3.set_title(f'Precision-Recall Curves\n{model_name}', fontweight='bold')
    ax3.legend(fontsize=7, loc='upper right')
    ax3.set_xlim([-0.01, 1.01]); ax3.set_ylim([-0.01, 1.01])

    plt.suptitle(
        f'IDRS — {model_name} Evaluation Suite | {split_name} Set | '
        f'F1={f1_mac:.4f} · AUC={roc_auc:.4f}',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / f'eval_{save_prefix}_{split_name.lower()}.png'
    plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"💾 Saved: {fname.name}")

    return results


print("✅ evaluate_model() toolkit ready.")

---
## 🌲 CELL 7 — Random Forest Baseline + Optuna Tuning

In [ ]:
# ============================================================
# IDRS Part 2 — Random Forest
# Phase 1: Quick baseline  →  Phase 2: Optuna optimisation
# ============================================================

# ─────────────────────────────────────────────────────────
# PHASE 1: Baseline RF (no tuning) — quick sanity check
# ─────────────────────────────────────────────────────────
print("🌲 Phase 1: Random Forest — Baseline")
t0 = time.time()

rf_baseline = RandomForestClassifier(
    n_estimators = 200,
    max_depth    = None,
    n_jobs       = -1,
    class_weight = 'balanced',
    random_state = SEED,
)
rf_baseline.fit(X_train_res, y_train_res)
print(f"   Training time: {time.time()-t0:.1f}s")

rf_base_metrics_val = evaluate_model(
    rf_baseline, X_val_scaled, y_val,
    model_name  = 'RF-Baseline',
    split_name  = 'Validation',
    save_prefix = 'rf_base',
    plot        = True,
)

# ─────────────────────────────────────────────────────────
# PHASE 2: Optuna Bayesian Hyperparameter Optimisation
# ─────────────────────────────────────────────────────────
print("\n🔬 Phase 2: Random Forest — Optuna Optimisation")
N_TRIALS_RF = 30  # increase for better results (50–100 for production)

def rf_objective(trial: optuna.Trial) -> float:
    params = {
        'n_estimators'          : trial.suggest_int('n_estimators', 100, 600, step=50),
        'max_depth'             : trial.suggest_categorical('max_depth',
                                     [None, 10, 20, 30, 40, 50]),
        'min_samples_split'     : trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf'      : trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features'          : trial.suggest_categorical('max_features',
                                     ['sqrt', 'log2', 0.3, 0.5, 0.7]),
        'bootstrap'             : trial.suggest_categorical('bootstrap', [True, False]),
        'max_samples'           : trial.suggest_float('max_samples', 0.5, 1.0)
                                     if trial.params.get('bootstrap', True) else None,
        'class_weight'          : trial.suggest_categorical('class_weight',
                                     ['balanced', 'balanced_subsample']),
        'n_jobs'                : -1,
        'random_state'          : SEED,
    }
    # max_samples only valid when bootstrap=True
    if not params['bootstrap']:
        params.pop('max_samples', None)

    model = RandomForestClassifier(**params)
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_val_scaled)
    # Optimise for macro-F1 (balances recall across rare attack classes)
    return f1_score(y_val, y_pred, average='macro', zero_division=0)


rf_study = optuna.create_study(
    direction  = 'maximize',
    study_name = 'IDRS_RF',
    sampler    = optuna.samplers.TPESampler(seed=SEED),
    pruner     = optuna.pruners.MedianPruner(n_warmup_steps=5),
)

print(f"   Running {N_TRIALS_RF} Optuna trials...")
t0 = time.time()

with tqdm(total=N_TRIALS_RF, desc="RF Optuna") as pbar:
    def rf_callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({'best_f1': f'{study.best_value:.4f}',
                          'trial': trial.number})
    rf_study.optimize(rf_objective, n_trials=N_TRIALS_RF,
                      callbacks=[rf_callback])

print(f"   Optuna complete in {time.time()-t0:.1f}s")
print(f"   Best val F1-macro : {rf_study.best_value:.4f}")
print(f"   Best params       : {rf_study.best_params}")

# ── Retrain best RF on full training set ─────────────────
best_rf_params = {**rf_study.best_params,
                  'n_jobs': -1, 'random_state': SEED}
if not best_rf_params.get('bootstrap', True):
    best_rf_params.pop('max_samples', None)

rf_best = RandomForestClassifier(**best_rf_params)
rf_best.fit(X_train_res, y_train_res)

# ── Evaluate on val + test ─────────────────────────────
rf_val_metrics = evaluate_model(
    rf_best, X_val_scaled, y_val,
    model_name='RF-Tuned', split_name='Validation', save_prefix='rf_tuned'
)
rf_test_metrics = evaluate_model(
    rf_best, X_test_scaled, y_test,
    model_name='RF-Tuned', split_name='Test', save_prefix='rf_tuned'
)

# Save
joblib.dump(rf_best, MODEL_DIR / 'rf_best.joblib')
print(f"\n💾 Saved: models/rf_best.joblib")

---
## ⚡ CELL 8 — XGBoost Baseline + Optuna Tuning

In [ ]:
# ============================================================
# IDRS Part 2 — XGBoost
# GPU-aware (uses 'hist' tree method; cuda if available)
# ============================================================
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
XGB_TREE_METHOD = 'hist'  # works on both CPU and GPU
XGB_DEVICE      = DEVICE
print(f"⚡ XGBoost device: {DEVICE}")

# ─────────────────────────────────────────────────────────
# PHASE 1: XGBoost Baseline
# ─────────────────────────────────────────────────────────
print("\n⚡ Phase 1: XGBoost — Baseline")
t0 = time.time()

xgb_baseline = xgb.XGBClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.1,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    use_label_encoder = False,
    eval_metric       = 'mlogloss',
    tree_method       = XGB_TREE_METHOD,
    device            = XGB_DEVICE,
    n_jobs            = -1,
    random_state      = SEED,
    verbosity         = 0,
)
xgb_baseline.fit(
    X_train_res, y_train_res,
    eval_set          = [(X_val_scaled, y_val)],
    verbose           = False,
)
print(f"   Training time: {time.time()-t0:.1f}s")

xgb_base_metrics = evaluate_model(
    xgb_baseline, X_val_scaled, y_val,
    model_name='XGB-Baseline', split_name='Validation',
    save_prefix='xgb_base'
)

# ─────────────────────────────────────────────────────────
# PHASE 2: XGBoost Optuna
# ─────────────────────────────────────────────────────────
print("\n🔬 Phase 2: XGBoost — Optuna Optimisation")
N_TRIALS_XGB = 35

def xgb_objective(trial: optuna.Trial) -> float:
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 800, step=50),
        'max_depth'        : trial.suggest_int('max_depth', 3, 12),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 20),
        'gamma'            : trial.suggest_float('gamma', 0, 5),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        'max_delta_step'   : trial.suggest_int('max_delta_step', 0, 10),
        'use_label_encoder': False,
        'eval_metric'      : 'mlogloss',
        'tree_method'      : XGB_TREE_METHOD,
        'device'           : XGB_DEVICE,
        'n_jobs'           : -1,
        'random_state'     : SEED,
        'verbosity'        : 0,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train_res, y_train_res,
        eval_set    = [(X_val_scaled, y_val)],
        verbose     = False,
    )
    y_pred = model.predict(X_val_scaled)
    return f1_score(y_val, y_pred, average='macro', zero_division=0)


xgb_study = optuna.create_study(
    direction  = 'maximize',
    study_name = 'IDRS_XGB',
    sampler    = optuna.samplers.TPESampler(seed=SEED),
)

t0 = time.time()
with tqdm(total=N_TRIALS_XGB, desc="XGB Optuna") as pbar:
    def xgb_callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({'best_f1': f'{study.best_value:.4f}'})
    xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_XGB,
                       callbacks=[xgb_callback])

print(f"   Optuna complete in {time.time()-t0:.1f}s")
print(f"   Best val F1-macro : {xgb_study.best_value:.4f}")
print(f"   Best params       : {xgb_study.best_params}")

# Retrain best XGBoost
best_xgb_params = {
    **xgb_study.best_params,
    'use_label_encoder': False, 'eval_metric': 'mlogloss',
    'tree_method': XGB_TREE_METHOD, 'device': XGB_DEVICE,
    'n_jobs': -1, 'random_state': SEED, 'verbosity': 0,
}
xgb_best = xgb.XGBClassifier(**best_xgb_params)
xgb_best.fit(X_train_res, y_train_res,
             eval_set=[(X_val_scaled, y_val)], verbose=False)

xgb_val_metrics  = evaluate_model(xgb_best, X_val_scaled, y_val,
                    model_name='XGB-Tuned', split_name='Validation',
                    save_prefix='xgb_tuned')
xgb_test_metrics = evaluate_model(xgb_best, X_test_scaled, y_test,
                    model_name='XGB-Tuned', split_name='Test',
                    save_prefix='xgb_tuned')

joblib.dump(xgb_best, MODEL_DIR / 'xgb_best.joblib')
print(f"\n💾 Saved: models/xgb_best.joblib")

---
## 💡 CELL 9 — LightGBM Baseline + Optuna Tuning

In [ ]:
# ============================================================
# IDRS Part 2 — LightGBM
# Fast gradient boosting; native categorical support
# ============================================================

# ─────────────────────────────────────────────────────────
# PHASE 1: LightGBM Baseline
# ─────────────────────────────────────────────────────────
print("💡 Phase 1: LightGBM — Baseline")
t0 = time.time()

lgbm_baseline = lgb.LGBMClassifier(
    n_estimators   = 400,
    max_depth      = -1,
    learning_rate  = 0.05,
    num_leaves     = 63,
    subsample      = 0.8,
    colsample_bytree= 0.8,
    class_weight   = 'balanced',
    n_jobs         = -1,
    random_state   = SEED,
    verbosity      = -1,
)
lgbm_baseline.fit(
    X_train_res, y_train_res,
    eval_set  = [(X_val_scaled, y_val)],
    callbacks = [lgb.early_stopping(50, verbose=False),
                 lgb.log_evaluation(-1)],
)
print(f"   Best iteration : {lgbm_baseline.best_iteration_}")
print(f"   Training time  : {time.time()-t0:.1f}s")

lgbm_base_metrics = evaluate_model(
    lgbm_baseline, X_val_scaled, y_val,
    model_name='LGBM-Baseline', split_name='Validation',
    save_prefix='lgbm_base'
)

# ─────────────────────────────────────────────────────────
# PHASE 2: LightGBM Optuna (LightGBM-native integration)
# ─────────────────────────────────────────────────────────
print("\n🔬 Phase 2: LightGBM — Optuna Optimisation")
N_TRIALS_LGBM = 35

def lgbm_objective(trial: optuna.Trial) -> float:
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth'         : trial.suggest_int('max_depth', 3, 15),
        'num_leaves'        : trial.suggest_int('num_leaves', 20, 300),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample'         : trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_samples' : trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        'min_split_gain'    : trial.suggest_float('min_split_gain', 0, 1),
        'bagging_freq'      : trial.suggest_int('bagging_freq', 0, 10),
        'path_smooth'       : trial.suggest_float('path_smooth', 0, 1),
        # ── Boosting type ──
        'boosting_type'     : trial.suggest_categorical('boosting_type',
                                 ['gbdt', 'dart', 'goss']),
        'class_weight'      : 'balanced',
        'n_jobs'            : -1,
        'random_state'      : SEED,
        'verbosity'         : -1,
    }
    # DART: disable early stopping
    callbacks = (
        [] if params['boosting_type'] == 'dart'
        else [lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
    )
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train_res, y_train_res,
        eval_set  = [(X_val_scaled, y_val)],
        callbacks = callbacks,
    )
    y_pred = model.predict(X_val_scaled)
    return f1_score(y_val, y_pred, average='macro', zero_division=0)


lgbm_study = optuna.create_study(
    direction  = 'maximize',
    study_name = 'IDRS_LGBM',
    sampler    = optuna.samplers.TPESampler(seed=SEED),
)

t0 = time.time()
with tqdm(total=N_TRIALS_LGBM, desc="LGBM Optuna") as pbar:
    def lgbm_callback_fn(study, trial):
        pbar.update(1)
        pbar.set_postfix({'best_f1': f'{study.best_value:.4f}'})
    lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS_LGBM,
                        callbacks=[lgbm_callback_fn])

print(f"   Optuna complete in {time.time()-t0:.1f}s")
print(f"   Best val F1-macro : {lgbm_study.best_value:.4f}")
print(f"   Best params       : {lgbm_study.best_params}")

# Retrain best LightGBM
best_lgbm_params = {
    **lgbm_study.best_params,
    'class_weight': 'balanced', 'n_jobs': -1,
    'random_state': SEED, 'verbosity': -1,
}
lgbm_best = lgb.LGBMClassifier(**best_lgbm_params)
lgbm_callbacks = (
    [] if best_lgbm_params.get('boosting_type') == 'dart'
    else [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)
lgbm_best.fit(
    X_train_res, y_train_res,
    eval_set  = [(X_val_scaled, y_val)],
    callbacks = lgbm_callbacks,
)

lgbm_val_metrics  = evaluate_model(lgbm_best, X_val_scaled, y_val,
                     model_name='LGBM-Tuned', split_name='Validation',
                     save_prefix='lgbm_tuned')
lgbm_test_metrics = evaluate_model(lgbm_best, X_test_scaled, y_test,
                     model_name='LGBM-Tuned', split_name='Test',
                     save_prefix='lgbm_tuned')

joblib.dump(lgbm_best, MODEL_DIR / 'lgbm_best.joblib')
print(f"\n💾 Saved: models/lgbm_best.joblib")

---
## 🎯 CELL 10 — Per-Class Threshold Optimisation

In [ ]:
# ============================================================
# IDRS Part 2 — Threshold Tuning for Critical Attack Classes
#
# Default threshold (0.5) optimises accuracy.
# For IDS, we want maximum recall on critical attacks
# (WebAttack, DoS, DDoS, Exploit) while keeping FPR < 5%.
#
# Strategy: Use validation set to find per-class thresholds
#           that maximise F-beta (beta=2, recall-weighted)
#           subject to FPR constraint.
# ============================================================

# Use the best-performing model on val set for threshold tuning
# (pick whichever achieved highest F1 on validation)
val_f1s = {
    'RF'  : rf_val_metrics['f1_macro'],
    'XGB' : xgb_val_metrics['f1_macro'],
    'LGBM': lgbm_val_metrics['f1_macro'],
}
best_model_name = max(val_f1s, key=val_f1s.get)
best_model_obj  = {'RF': rf_best, 'XGB': xgb_best, 'LGBM': lgbm_best}[best_model_name]

print(f"🎯 Threshold tuning on: {best_model_name} (val F1={val_f1s[best_model_name]:.4f})")

# Critical classes: higher beta = more weight on recall
CRITICAL_CLASSES = {'WebAttack', 'DoS', 'DDoS', 'Exploit', 'BruteForce', 'Botnet'}
BETA = 2.0   # F-beta: recall is twice as important as precision
MAX_FPR = 0.05  # maximum tolerable false positive rate per class

y_prob_val  = best_model_obj.predict_proba(X_val_scaled)
y_bin_val   = label_binarize(y_val, classes=list(range(N_CLASSES)))

thresholds = {}

fig, axes = plt.subplots(
    (N_CLASSES + 3) // 4, 4,
    figsize=(20, ((N_CLASSES + 3) // 4) * 4.5)
)
axes = np.array(axes).flatten()

for c, cls_name in enumerate(CLASS_NAMES):
    ax     = axes[c]
    probs  = y_prob_val[:, c]
    labels = y_bin_val[:, c]

    if labels.sum() == 0:
        thresholds[cls_name] = 0.5
        ax.set_visible(False)
        continue

    # Sweep thresholds from 0.05 to 0.95
    thresh_range = np.linspace(0.05, 0.95, 200)
    f_betas, fprs, recalls, precisions = [], [], [], []

    for t in thresh_range:
        preds_t = (probs >= t).astype(int)
        tp = ((preds_t == 1) & (labels == 1)).sum()
        fp = ((preds_t == 1) & (labels == 0)).sum()
        fn = ((preds_t == 0) & (labels == 1)).sum()
        tn = ((preds_t == 0) & (labels == 0)).sum()
        prec = tp / (tp + fp + 1e-9)
        rec  = tp / (tp + fn + 1e-9)
        fpr  = fp / (fp + tn + 1e-9)
        fb   = (1 + BETA**2) * prec * rec / (BETA**2 * prec + rec + 1e-9)
        f_betas.append(fb)
        fprs.append(fpr)
        recalls.append(rec)
        precisions.append(prec)

    f_betas  = np.array(f_betas)
    fprs     = np.array(fprs)

    # Find best threshold: max F-beta subject to FPR < MAX_FPR
    # For critical classes: relax FPR constraint slightly
    fpr_limit = MAX_FPR * 2 if cls_name in CRITICAL_CLASSES else MAX_FPR
    valid_mask = fprs <= fpr_limit

    if valid_mask.sum() > 0:
        best_idx  = np.argmax(f_betas * valid_mask)
    else:
        best_idx  = np.argmax(f_betas)  # relax if no valid threshold

    best_thresh = float(thresh_range[best_idx])
    thresholds[cls_name] = best_thresh

    # ── Plot ─────────────────────────────────────────────
    color = PALETTE.get(cls_name, '#95a5a6')
    ax.plot(thresh_range, f_betas,    lw=2, color=color,    label=f'F{BETA:.0f}-score')
    ax.plot(thresh_range, recalls,    lw=1.5, color='#27ae60', ls='--', label='Recall')
    ax.plot(thresh_range, precisions, lw=1.5, color='#2980b9', ls=':',  label='Precision')
    ax.plot(thresh_range, fprs,       lw=1.2, color='#e74c3c', ls='-.', label='FPR')
    ax.axvline(best_thresh, color='black', lw=2, ls='--',
               label=f'Best t={best_thresh:.2f}')
    ax.axhline(fpr_limit, color='#e74c3c', lw=0.8, alpha=0.5,
               label=f'FPR limit={fpr_limit:.2f}')

    critical_marker = "🔴" if cls_name in CRITICAL_CLASSES else "🟢"
    ax.set_title(f'{critical_marker} {cls_name}\nt*={best_thresh:.2f} '
                 f'F{BETA:.0f}={f_betas[best_idx]:.3f}',
                 fontsize=8, fontweight='bold')
    ax.set_xlabel('Threshold', fontsize=7)
    ax.set_xlim([0.05, 0.95])
    ax.set_ylim([-0.02, 1.05])
    ax.legend(fontsize=6, loc='center left')

for idx in range(N_CLASSES, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(
    f'Per-Class Threshold Optimisation — {best_model_name}\n'
    f'Objective: max F{BETA:.0f}-score | FPR constraint: {MAX_FPR*100:.0f}% '
    f'(critical: {MAX_FPR*200:.0f}%)',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'threshold_tuning.png', bbox_inches='tight', dpi=150)
plt.show()

print("\n🎯 Optimised thresholds:")
for cls, t in sorted(thresholds.items()):
    flag = '🔴 CRITICAL' if cls in CRITICAL_CLASSES else '🟢 normal'
    print(f"   {cls:<14} t* = {t:.3f}  {flag}")

# Save thresholds
with open(MODEL_DIR / 'thresholds.json', 'w') as f:
    json.dump({'thresholds': thresholds,
               'tuned_on_model': best_model_name,
               'beta': BETA, 'max_fpr': MAX_FPR}, f, indent=2)
print(f"\n💾 Saved: models/thresholds.json")

---
## 🔍 CELL 11 — SHAP Global Feature Importance

In [ ]:
# ============================================================
# IDRS Part 2 — SHAP Global Feature Importance
# Uses the best-performing model (RF or LightGBM recommended).
# ============================================================

print(f"🔍 Computing SHAP values for: {best_model_name}")
print("   (This may take 1–3 minutes depending on dataset size)")

# Use a stratified sample for SHAP (speed)
SHAP_SAMPLE = min(2000, len(X_val_scaled))
shap_idx = StratifiedShuffleSplit(
    n_splits=1, test_size=SHAP_SAMPLE, random_state=SEED
).split(X_val_scaled, y_val)
_, shap_sample_idx = next(shap_idx)

X_shap = X_val_scaled[shap_sample_idx]
y_shap = y_val[shap_sample_idx]

# ── Build SHAP explainer ──────────────────────────────────
t0 = time.time()
if best_model_name in ['RF', 'XGB', 'LGBM']:
    explainer   = shap.TreeExplainer(best_model_obj)
    shap_values = explainer.shap_values(X_shap)  # list of arrays per class
else:
    explainer   = shap.KernelExplainer(
        best_model_obj.predict_proba,
        shap.sample(X_shap, 100)
    )
    shap_values = explainer.shap_values(X_shap)

print(f"   SHAP computation: {time.time()-t0:.1f}s")

# ── Handle output shape ───────────────────────────────────
# XGBoost: shap_values shape can be (n_samples, n_features, n_classes)
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    # (n, features, classes) → list of (n, features)
    shap_values = [shap_values[:, :, c] for c in range(N_CLASSES)]

shap_values_list = shap_values if isinstance(shap_values, list) else [shap_values]

# ── 1. Beeswarm plot — most important class (by abs SHAP) ─
# Use the attack class with highest mean |SHAP|
class_shap_means = [
    np.abs(sv).mean()
    for sv in shap_values_list
    if isinstance(sv, np.ndarray)
]
focus_class_idx = int(np.argmax(class_shap_means))
focus_class_name = CLASS_NAMES[focus_class_idx] if focus_class_idx < N_CLASSES else 'Class0'

print(f"\n   Focus class for beeswarm: {focus_class_name} (highest mean |SHAP|)")

sv_focus = shap_values_list[focus_class_idx]

fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(
    sv_focus,
    features           = X_shap,
    feature_names      = FEATURE_COLS,
    max_display        = 20,
    show               = False,
    plot_type          = 'dot',
    color_bar_label    = 'Feature value',
)
plt.title(f'SHAP Beeswarm — {best_model_name} · Class: {focus_class_name}\n'
          f'(Top 20 features by mean |SHAP| impact)',
          fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_beeswarm.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"💾 Saved: outputs/shap_beeswarm.png")

---
## 🔍 CELL 12 — SHAP: Multi-Class Bar + Decision Plots

In [ ]:
# ============================================================
# IDRS Part 2 — SHAP: Global bar (all classes) + Local plots
# ============================================================

# ── 1. Global mean |SHAP| — all classes stacked ──────────
if isinstance(shap_values_list[0], np.ndarray):
    mean_abs_shap = np.stack([
        np.abs(sv).mean(axis=0) for sv in shap_values_list
    ], axis=0)  # shape: (n_classes, n_features)

    # Top 20 features by average importance across all classes
    global_importance = mean_abs_shap.mean(axis=0)
    top20_idx         = np.argsort(global_importance)[::-1][:20]
    top20_feats       = [FEATURE_COLS[i] for i in top20_idx]
    top20_vals        = mean_abs_shap[:, top20_idx]  # (n_classes, 20)

    fig, ax = plt.subplots(figsize=(14, 7))
    x_pos   = np.arange(len(top20_feats))
    bottom  = np.zeros(len(top20_feats))

    for c, cls_name in enumerate(CLASS_NAMES):
        color  = PALETTE.get(cls_name, '#95a5a6')
        vals   = top20_vals[c]
        ax.bar(x_pos, vals, bottom=bottom, color=color,
               label=cls_name, alpha=0.85, width=0.75)
        bottom += vals

    ax.set_xticks(x_pos)
    ax.set_xticklabels(top20_feats, rotation=40, ha='right', fontsize=8)
    ax.set_ylabel('Mean |SHAP| Value (stacked across classes)')
    ax.set_title(f'Global Feature Importance — {best_model_name}\n'
                 f'Stacked per-class SHAP contributions (Top 20 features)',
                 fontweight='bold')
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'shap_global_bar.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f"💾 Saved: outputs/shap_global_bar.png")

    # ── Save feature importance ranking ──────────────────
    feat_importance_df = pd.DataFrame({
        'feature'          : FEATURE_COLS,
        'global_importance': global_importance,
        **{f'shap_{cls}': mean_abs_shap[c]
           for c, cls in enumerate(CLASS_NAMES)}
    }).sort_values('global_importance', ascending=False)
    feat_importance_df.to_csv(OUTPUT_DIR / 'shap_feature_importance.csv', index=False)
    print(f"💾 Saved: outputs/shap_feature_importance.csv")
    print(f"\n   Top 10 globally important features:")
    print(feat_importance_df[['feature','global_importance']].head(10).to_string(index=False))

# ── 2. Local SHAP: Force plot for one attack sample ──────
# Pick first sample predicted as an attack class
attack_mask = y_shap != CLASS_TO_INT.get('BENIGN', 0)
if attack_mask.sum() > 0:
    sample_idx = np.where(attack_mask)[0][0]
    true_cls   = INT_TO_CLASS.get(int(y_shap[sample_idx]), 'Unknown')
    pred_cls_id= int(best_model_obj.predict(X_shap[[sample_idx]])[0])
    pred_cls   = INT_TO_CLASS.get(pred_cls_id, 'Unknown')

    print(f"\n   🔎 Local SHAP — Sample idx={sample_idx}")
    print(f"      True class : {true_cls}")
    print(f"      Pred class : {pred_cls}")

    if isinstance(shap_values_list[0], np.ndarray):
        local_sv   = shap_values_list[pred_cls_id][sample_idx]
        base_val   = explainer.expected_value
        if isinstance(base_val, (list, np.ndarray)):
            base_val = base_val[pred_cls_id]

        # Waterfall plot for local explanation
        shap_exp = shap.Explanation(
            values        = local_sv,
            base_values   = base_val,
            data          = X_shap[sample_idx],
            feature_names = FEATURE_COLS,
        )
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(shap_exp, max_display=15, show=False)
        plt.title(f'Local SHAP Waterfall — True: {true_cls} | Pred: {pred_cls}',
                  fontweight='bold')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'shap_local_waterfall.png', bbox_inches='tight', dpi=150)
        plt.show()
        print(f"💾 Saved: outputs/shap_local_waterfall.png")

---
## 🏆 CELL 13 — Cross-Model Comparison Dashboard

In [ ]:
# ============================================================
# IDRS Part 2 — Cross-Model Comparison Table & Radar Chart
# ============================================================

# ── Aggregate all model metrics ───────────────────────────
ALL_METRICS = [
    rf_test_metrics,
    xgb_test_metrics,
    lgbm_test_metrics,
]

metrics_df = pd.DataFrame(ALL_METRICS)
display_cols = ['model','accuracy','balanced_acc','mcc',
                'f1_macro','f1_weighted','precision_macro',
                'recall_macro','roc_auc','pr_auc']
metrics_df = metrics_df[[c for c in display_cols if c in metrics_df.columns]]
metrics_df = metrics_df.set_index('model')

print("\n🏆 Classical ML Comparison (Test Set)")
print("="*80)
print(metrics_df.to_string())

best_overall = metrics_df['f1_macro'].idxmax()
print(f"\n   🥇 Best model (F1-macro): {best_overall}")

# ── Save comparison table ─────────────────────────────────
metrics_df.to_csv(OUTPUT_DIR / 'classical_ml_comparison.csv')

# ── Radar Chart ──────────────────────────────────────────
radar_metrics = ['accuracy','balanced_acc','f1_macro','precision_macro',
                 'recall_macro','roc_auc','pr_auc']
radar_labels  = ['Accuracy','Bal.Acc','F1 Macro','Precision','Recall','ROC-AUC','PR-AUC']
radar_metrics = [m for m in radar_metrics if m in metrics_df.columns]
radar_labels  = radar_labels[:len(radar_metrics)]

N_RADAR   = len(radar_metrics)
angles    = np.linspace(0, 2*np.pi, N_RADAR, endpoint=False).tolist()
angles   += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'projection': 'polar'})
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

model_colors = ['#3498db', '#e74c3c', '#2ecc71']
for idx, (model_name_r, row) in enumerate(metrics_df.iterrows()):
    values  = [row[m] for m in radar_metrics]
    values += values[:1]
    color   = model_colors[idx % len(model_colors)]
    ax.plot(angles, values, 'o-', linewidth=2.5, color=color, label=model_name_r)
    ax.fill(angles, values, alpha=0.12, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8)
ax.grid(True, alpha=0.5)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=11)
ax.set_title('Classical ML Models — Test Set Performance Radar',
             pad=20, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison_radar.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"💾 Saved: outputs/model_comparison_radar.png")

# ── Metric bar comparison ─────────────────────────────────
fig, axes = plt.subplots(1, len(radar_metrics), figsize=(22, 5))
for idx, (metric, label) in enumerate(zip(radar_metrics, radar_labels)):
    ax     = axes[idx]
    models = metrics_df.index.tolist()
    vals   = metrics_df[metric].values
    colors = [model_colors[i % 3] for i in range(len(models))]
    bars   = ax.bar(models, vals, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_ylim(max(0, vals.min()-0.1), min(1.0, vals.max()+0.08))
    ax.set_title(label, fontweight='bold', fontsize=9)
    ax.tick_params(axis='x', rotation=20, labelsize=8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

plt.suptitle('Classical ML Metric Comparison — Test Set', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison_bars.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"💾 Saved: outputs/model_comparison_bars.png")

---
## 📊 CELL 14 — Optuna Visualisation & Hyperparameter Analysis

In [ ]:
# ============================================================
# IDRS Part 2 — Optuna Study Visualisations
# ============================================================

studies = {
    'RandomForest' : rf_study,
    'XGBoost'      : xgb_study,
    'LightGBM'     : lgbm_study,
}

fig, axes = plt.subplots(1, 3, figsize=(21, 5))

for idx, (name, study) in enumerate(studies.items()):
    ax = axes[idx]

    trial_nums = [t.number for t in study.trials if t.value is not None]
    trial_vals = [t.value  for t in study.trials if t.value is not None]

    # Running best
    best_so_far = []
    cur_best = -np.inf
    for v in trial_vals:
        cur_best = max(cur_best, v)
        best_so_far.append(cur_best)

    color = model_colors[idx]
    ax.scatter(trial_nums, trial_vals, alpha=0.5, s=25, color=color, zorder=3)
    ax.plot(trial_nums, best_so_far, lw=2.5, color='black',
            label=f'Best: {best_so_far[-1]:.4f}', zorder=4)
    ax.fill_between(trial_nums, best_so_far, alpha=0.15, color=color)
    ax.set_title(f'{name}\nOptuna Optimisation History', fontweight='bold')
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('Val F1-macro')
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=max(0, min(trial_vals)-0.05))

plt.suptitle('Optuna Hyperparameter Search — F1-macro Progression',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'optuna_history.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"💾 Saved: outputs/optuna_history.png")

# ── Best params summary ───────────────────────────────────
print("\n📋 Best Hyperparameters Summary:")
print("-" * 60)
for name, study in studies.items():
    print(f"\n{name}  (F1={study.best_value:.4f}):")
    for k, v in study.best_params.items():
        print(f"   {k:<30} = {v}")

# Save best params
best_params_all = {
    name: {'best_f1': study.best_value, 'params': study.best_params}
    for name, study in studies.items()
}
with open(MODEL_DIR / 'best_hyperparams.json', 'w') as f:
    json.dump(best_params_all, f, indent=2, default=str)
print(f"\n💾 Saved: models/best_hyperparams.json")

---
## 🔁 CELL 15 — Stratified K-Fold Cross-Validation (Best Model)

In [ ]:
# ============================================================
# IDRS Part 2 — 5-Fold Stratified Cross-Validation
# Validates generalisation of the best model.
# Run on the FULL (pre-SMOTE) scaled dataset to avoid leakage.
# ============================================================

print(f"🔁 5-Fold Stratified Cross-Validation — {best_model_name}")
print("   (SMOTE applied per-fold inside CV to prevent leakage)")

K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)

X_full = scaler.transform(X.values)
y_full = y

cv_metrics = {metric: [] for metric in
              ['accuracy','f1_macro','f1_weighted','roc_auc','recall_macro']}

for fold, (train_idx, val_idx) in enumerate(
    tqdm(skf.split(X_full, y_full), total=K, desc=f"{K}-Fold CV")):

    X_fold_tr, X_fold_val = X_full[train_idx], X_full[val_idx]
    y_fold_tr, y_fold_val = y_full[train_idx], y_full[val_idx]

    # Apply SMOTE inside fold (prevent leakage)
    fold_counts = pd.Series(y_fold_tr).value_counts()
    fold_ratio  = fold_counts.max() / fold_counts.min()

    if fold_ratio > 3 and fold_counts.min() > K_NEIGHBORS:
        fold_target_n = max(int(fold_counts.max() * 0.2), fold_counts.min())
        fold_strategy = {
            cid: max(fold_target_n, cnt)
            for cid, cnt in fold_counts.items()
            if cnt < fold_target_n and cnt > K_NEIGHBORS
        }
        if fold_strategy:
            smote_fold = BorderlineSMOTE(
                sampling_strategy=fold_strategy,
                k_neighbors=K_NEIGHBORS, random_state=SEED
            )
            try:
                X_fold_tr, y_fold_tr = smote_fold.fit_resample(X_fold_tr, y_fold_tr)
            except Exception:
                pass  # skip if SMOTE fails on this fold

    # Clone & retrain best model
    fold_model = deepcopy(best_model_obj)
    fold_model.fit(X_fold_tr, y_fold_tr)

    y_pred_fold = fold_model.predict(X_fold_val)
    y_prob_fold = fold_model.predict_proba(X_fold_val)

    cv_metrics['accuracy'].append(accuracy_score(y_fold_val, y_pred_fold))
    cv_metrics['f1_macro'].append(f1_score(y_fold_val, y_pred_fold,
                                           average='macro', zero_division=0))
    cv_metrics['f1_weighted'].append(f1_score(y_fold_val, y_pred_fold,
                                             average='weighted', zero_division=0))
    cv_metrics['recall_macro'].append(recall_score(y_fold_val, y_pred_fold,
                                                   average='macro', zero_division=0))
    try:
        y_bin_fold = label_binarize(y_fold_val, classes=list(range(N_CLASSES)))
        cv_metrics['roc_auc'].append(
            roc_auc_score(y_bin_fold, y_prob_fold, average='macro', multi_class='ovr')
        )
    except Exception:
        cv_metrics['roc_auc'].append(float('nan'))

print(f"\n   {K}-Fold CV Results — {best_model_name}")
print(f"   {'Metric':<20} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("   " + "-"*52)
cv_summary = {}
for metric, vals in cv_metrics.items():
    arr = np.array(vals)
    cv_summary[metric] = {'mean': arr.mean(), 'std': arr.std(),
                           'min': arr.min(), 'max': arr.max()}
    print(f"   {metric:<20} {arr.mean():>8.4f} {arr.std():>8.4f} "
          f"{arr.min():>8.4f} {arr.max():>8.4f}")

# Box plots
fig, axes = plt.subplots(1, len(cv_metrics), figsize=(18, 5))
for idx, (metric, vals) in enumerate(cv_metrics.items()):
    ax = axes[idx]
    ax.boxplot(vals, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.7),
               medianprops=dict(color='black', linewidth=2.5))
    ax.scatter([1]*len(vals), vals, color='#2c3e50', zorder=5, s=60, alpha=0.8)
    ax.set_title(metric.replace('_',' ').title(), fontsize=9, fontweight='bold')
    ax.set_xlim(0.5, 1.5)
    ax.set_xticks([])
    mean_v = np.mean(vals)
    ax.text(1, ax.get_ylim()[0], f'μ={mean_v:.4f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold', color='#e74c3c')

plt.suptitle(f'{K}-Fold Cross-Validation — {best_model_name} '
             f'(SMOTE per-fold, no leakage)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cv_results.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"💾 Saved: outputs/cv_results.png")

with open(OUTPUT_DIR / 'cv_summary.json', 'w') as f:
    json.dump({'model': best_model_name, 'k': K, 'results': cv_summary}, f, indent=2)

---
## 💾 CELL 16 — Save All Artifacts & Part 2 Summary

In [ ]:
# ============================================================
# IDRS Part 2 — Save All Artifacts & Final Summary
# ============================================================

# ── 1. Save final model registry ─────────────────────────
model_registry = {
    'RandomForest': {
        'path'       : str(MODEL_DIR / 'rf_best.joblib'),
        'test_f1'    : rf_test_metrics['f1_macro'],
        'test_auc'   : rf_test_metrics['roc_auc'],
        'best_params': rf_study.best_params,
    },
    'XGBoost': {
        'path'       : str(MODEL_DIR / 'xgb_best.joblib'),
        'test_f1'    : xgb_test_metrics['f1_macro'],
        'test_auc'   : xgb_test_metrics['roc_auc'],
        'best_params': xgb_study.best_params,
    },
    'LightGBM': {
        'path'       : str(MODEL_DIR / 'lgbm_best.joblib'),
        'test_f1'    : lgbm_test_metrics['f1_macro'],
        'test_auc'   : lgbm_test_metrics['roc_auc'],
        'best_params': lgbm_study.best_params,
    },
    'best_model'    : best_model_name,
    'feature_cols'  : FEATURE_COLS,
    'class_names'   : CLASS_NAMES,
    'n_features'    : len(FEATURE_COLS),
    'n_classes'     : N_CLASSES,
    'thresholds_path': str(MODEL_DIR / 'thresholds.json'),
    'scaler_path'   : str(MODEL_DIR / 'scaler.joblib'),
    'created_at'    : datetime.now().isoformat(),
}

with open(MODEL_DIR / 'model_registry.json', 'w') as f:
    json.dump(model_registry, f, indent=2, default=str)

# ── 2. Save split indices for Part 3 reproducibility ─────
split_meta = {
    'train_n'     : len(X_train_res),
    'val_n'       : len(X_val_scaled),
    'test_n'      : len(X_test_scaled),
    'smote_type'  : SMOTE_TYPE if SMOTE_APPLIED else 'none',
    'split_type'  : split_type,
    'seed'        : SEED,
}
with open(OUTPUT_DIR / 'split_meta.json', 'w') as f:
    json.dump(split_meta, f, indent=2)

# Save scaled arrays for Part 3 (deep learning)
np.save(OUTPUT_DIR / 'X_train_res.npy', X_train_res)
np.save(OUTPUT_DIR / 'y_train_res.npy', y_train_res)
np.save(OUTPUT_DIR / 'X_val_scaled.npy', X_val_scaled)
np.save(OUTPUT_DIR / 'y_val.npy',        y_val)
np.save(OUTPUT_DIR / 'X_test_scaled.npy',X_test_scaled)
np.save(OUTPUT_DIR / 'y_test.npy',       y_test)

print("✅ All Part 2 artifacts saved.")
print("\n📦 Models:")
for f in sorted(MODEL_DIR.glob("*")):
    print(f"   • {f.name}  ({f.stat().st_size/1024:.1f} KB)")

# ── 3. Part 2 Summary Dashboard ───────────────────────────
fig = plt.figure(figsize=(20, 9))
fig.patch.set_facecolor('#0d1117')

ax = fig.add_subplot(111)
ax.set_facecolor('#0d1117')
ax.axis('off')

ax.text(0.5, 0.97, '🛡️  IDRS PART 2 — SUMMARY',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=20, fontweight='bold', color='white')
ax.text(0.5, 0.90, 'Preprocessing · Classical ML · Optuna Tuning · SHAP Explainability · CV',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=11, color='#8b949e')

# Three model score blocks
block_data = [
    ('RandomForest',  rf_test_metrics,   '#3498db', 0.05),
    ('XGBoost',       xgb_test_metrics,  '#e74c3c', 0.37),
    ('LightGBM',      lgbm_test_metrics, '#2ecc71', 0.69),
]
for model_lbl, met, color, x_off in block_data:
    crown = '👑 ' if model_lbl == best_model_name else '   '
    ax.text(x_off+0.13, 0.80, f'{crown}{model_lbl}',
            transform=ax.transAxes, ha='center',
            fontsize=12, fontweight='bold', color=color)
    lines = [
        f"F1 Macro   : {met['f1_macro']:.4f}",
        f"ROC-AUC    : {met['roc_auc']:.4f}",
        f"PR-AUC     : {met['pr_auc']:.4f}",
        f"Accuracy   : {met['accuracy']:.4f}",
        f"Recall Macro: {met['recall_macro']:.4f}",
    ]
    for i, line in enumerate(lines):
        ax.text(x_off+0.13, 0.72 - i*0.07, line,
                transform=ax.transAxes, ha='center',
                fontsize=9, color='white', fontfamily='monospace')

# Pipeline summary
summary_items = [
    ("SMOTE",       SMOTE_TYPE if SMOTE_APPLIED else 'None applied'),
    ("Split",       split_type),
    ("Optuna RF",   f"{N_TRIALS_RF} trials → F1={rf_study.best_value:.4f}"),
    ("Optuna XGB",  f"{N_TRIALS_XGB} trials → F1={xgb_study.best_value:.4f}"),
    ("Optuna LGBM", f"{N_TRIALS_LGBM} trials → F1={lgbm_study.best_value:.4f}"),
    ("Best Model",  best_model_name),
    ("CV F1 Mean",  f"{cv_summary['f1_macro']['mean']:.4f} ± {cv_summary['f1_macro']['std']:.4f}"),
    ("Thresholds",  "Per-class F2-optimised"),
    ("SHAP",        f"TreeExplainer · {SHAP_SAMPLE} samples"),
    ("Features",    str(len(FEATURE_COLS))),
]

for i, (k, v) in enumerate(summary_items):
    col = 0.05 if i < 5 else 0.55
    row = i % 5
    y_p = 0.35 - row * 0.065
    ax.text(col,      y_p, f"{k}:", transform=ax.transAxes,
            fontsize=9, color='#8b949e', fontweight='bold')
    ax.text(col+0.16, y_p, v, transform=ax.transAxes,
            fontsize=9, color='#f0f6fc')

ax.text(0.5, 0.03,
        '→ Part 3: 1D-CNN · BiLSTM · Hybrid CNN-LSTM Deep Learning Detectors',
        transform=ax.transAxes, ha='center', va='bottom',
        fontsize=9, color='#f39c12', style='italic')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'part2_summary.png', bbox_inches='tight',
            dpi=150, facecolor='#0d1117')
plt.show()

logger.info("Part 2 complete. Best model: %s | F1=%.4f | AUC=%.4f",
            best_model_name,
            model_registry[best_model_name.replace(' ','')]['test_f1'],
            model_registry[best_model_name.replace(' ','')]['test_auc']
            if best_model_name in model_registry else 0)

print("\n" + "="*60)
print("✅ PART 2 COMPLETE")
print(f"   Best classical model : {best_model_name}")
print(f"   → Proceed to IDRS_Part3_DeepLearning.ipynb")
print("="*60)

---
## ✅ Part 2 — Completion Checklist

| Task | Status |
|---|---|
| Load & validate Part 1 processed dataset | ✅ |
| Feature matrix construction with column audit | ✅ |
| Stratified / temporal train-val-test split | ✅ |
| StandardScaler (fit on train only) | ✅ |
| Adaptive SMOTE strategy (BorderlineSMOTE / SMOTETomek) | ✅ |
| Evaluation toolkit (accuracy, F1, ROC-AUC, PR-AUC, MCC, confusion matrix) | ✅ |
| Random Forest: baseline + 30-trial Optuna | ✅ |
| XGBoost: baseline + 35-trial Optuna (GPU-aware) | ✅ |
| LightGBM: baseline + 35-trial Optuna (early stopping, DART support) | ✅ |
| Per-class threshold tuning (F2-score + FPR constraint) | ✅ |
| SHAP TreeExplainer: beeswarm + global bar + local waterfall | ✅ |
| Cross-model radar chart + bar comparison | ✅ |
| Optuna history visualisation | ✅ |
| 5-Fold stratified CV with per-fold SMOTE (no leakage) | ✅ |
| All artifacts saved (joblib + json + npy) | ✅ |

---
### ➡️ Next: `IDRS_Part3_DeepLearning.ipynb`
- **1D-CNN** on network flow features  
- **BiLSTM** for temporal traffic patterns  
- **Hybrid CNN-LSTM** (main deep model)  
- Training loop: early stopping · LR scheduling · gradient clipping  
- Full deep vs classical comparison table